# 13 · Hints from the full-resolution frame

Every model so far works from what the client has: the low resolution frame the stream
delivered. The host has more - the full-resolution screen it shrank - and it could send a little
of what it knows along with the video. Notebook 12 is the reason to ask: its predictor, looking
at the LR alone, did not learn which way an edge runs, and the host can simply see it.

A **hint** is a few bits the host works out from the HR frame and sends with the stream. This
notebook trains no network. It measures what three kinds of hint are worth, with closed-form
fits and the models already published:

1. **A content label per tile** - a screen, or a photo or video - so the client runs the model
   that is best at it. Notebook 10's table is best on screens, notebook 4's model on photos and
   video.
2. **Filters fitted by the host, on bicubic.** The client sorts every LR pixel into a class by
   the direction and strength of its edges; the host fits, per class, the 5x5 filter that best
   turns its LR into its own HR frame, and sends the coefficients. Inside a codec, this is what
   VVC's adaptive loop filter does.
3. **The same on top of every published model** - a post-filter on its output, the way a
   codec's loop filter works on its own reconstruction.

**Bits are the price.** The host's default stream is 0.46 bits per LR pixel (8 Mbit/s at
960x540, 30 fps), about 238 kbit a frame. Every hint is counted in bits per update and as a
share of one frame, and set against what the same bits would buy spent on H.264 instead -
measured here too, from how the patch set's quality moves with its bitrate, which was drawn at
random per source.

**Held out, as on a stream.** A source's filters are fitted on frames it is not scored on: every
patch is a clip of 8 frames, and the host fits on the other 7 frames of a source's 16 patches,
then the filters are scored on the frame every notebook scores. Filters sent for a scene and
used on a frame they were not fitted to - not fitted to the very pixels they are scored on.

Steps:

1. **Data** - the patch set, what each patch shows, and how each source was streamed.
2. **The published graphs** - every model to route between, run as it ships, 8 patches to a
   client tile, and checked against what its notebook measured.
3. **Routing** - by the true label, by a classifier on the HR (the host) and on the LR (the
   client, free), and by the best model per tile.
4. **Filters fitted by the host** - one filter or 25 classes, fitted on training data (no bits)
   or per source from the HR, on bicubic and on each published model.
5. **What the bits are worth** - the same bits in H.264, and every hint side by side.

Needs: `00_data_preprocess.ipynb`, and the graphs notebooks 4, 10 and 12 published in
`dist/upscale/`. Writes nothing.

In [1]:
"""Settings."""

import math

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

from helpers import degrade, metrics, models, patches, postfilter, training, webexport
from helpers.patches import PatchPairs
from helpers.postfilter import luma
from helpers.training import DATA_DIR, DEVICE

GRAPHS = {   # the models a tile can be routed to, as published, and the ssim-y their notebooks measured on this set
    "notebook 4": ("upscale_strategy_luma_24_2_blocks_tile320x180.onnx", 0.9292),   # best on photos and video
    "SDY x2 table": ("upscale_lut_sdy_x2_17_tile320x180.onnx", 0.9343),             # notebook 10: best on screens
    "DirNet D": ("upscale_dirnet_full_tile320x180.onnx", 0.9303),                   # notebook 12: next on screens
}
BEST = "notebook 4"          # where a tile goes unless it is a screen

STREAM_BPP = degrade.STREAM_BPP   # the host's default: 8 Mbit/s at 960x540, 30 fps
FRAME_LR = (960, 540)             # the LR frame a hint's bits are counted against
TAPS = 5                          # a filter reads the 5x5 pixels around the one it makes
LEVELS = 5                        # classes: no direction, across, down, the two diagonals; x 5 levels of activity
RIDGE = 1.0                       # a host's fit is pulled toward the zero-bit filter with the weight of one pixel
MIN_PIXELS = 2000                 # a class a source shows less often keeps the zero-bit filter
COEFFICIENT_BITS = 10             # per coefficient, sent as a difference from the zero-bit filter
STEP_BITS = 16                    # per filter: the step its coefficients are counted in
TRAIN_STEP = 2                    # every 2nd training patch fits the classifiers and the zero-bit filters

CLASSES = postfilter.DIRECTIONS * LEVELS
FRAME_BITS = STREAM_BPP * FRAME_LR[0] * FRAME_LR[1]
TILES_PER_FRAME = FRAME_LR[0] * FRAME_LR[1] / 64 ** 2   # 64x64 LR tiles, one patch's core
training.seed_everything(0)
print(f"one frame of the default stream: {FRAME_BITS / 1e3:.0f} kbit, {TILES_PER_FRAME:.0f} tiles of 64x64")

one frame of the default stream: 238 kbit, 127 tiles of 64x64


## 1. Data

The validation frame of every patch, as every notebook scores it, with what its source shows
and the bitrate it was streamed at.

In [2]:
manifest = training.load_manifest()
training.describe(manifest)
SCALE, HALO, CORE, PATCH = manifest["scale"], manifest["halo"], manifest["patch_lr"], manifest["input_lr"]
trim = training.trimmer(manifest)
val_lr, val_hr = patches.load(DATA_DIR, "val", manifest)   # uint8, (N, 72, 72, 3) and (N, 128, 128, 3)
sources = manifest["splits"]["val"]["sources"]


def content(source):
    """What a source shows: a drawn desktop screen, a photo (as a moving clip) or a video."""
    if source["dataset"] == "screens":
        return "screen"
    return "photo" if "motion" in source else "video"


def per_patch(split):
    """For every patch of a split: the index of its source, and whether it shows a screen."""
    split_sources = manifest["splits"][split]["sources"]
    source = np.zeros(split_sources[-1]["patches"][1], np.int64)
    for k, s in enumerate(split_sources):
        source[s["patches"][0]:s["patches"][1]] = k
    return source, np.array([split_sources[k]["dataset"] == "screens" for k in source])


def to_tensor(images):
    """uint8 (N, h, w, 3) -> float (N, 3, h, w) in [0, 1] on DEVICE."""
    return torch.from_numpy(np.ascontiguousarray(images)).to(DEVICE).permute(0, 3, 1, 2).float() / 255


val_source, val_screen = per_patch("val")
kinds = np.array([content(sources[k]) for k in val_source])
by_content = {kind: np.flatnonzero(kinds == kind) for kind in ("photo", "video", "screen")}
print("val by content: " + ", ".join(f"{kind} {len(index):,}" for kind, index in by_content.items()))

device  cuda NVIDIA GeForce RTX 3090
scale   2
patch   72 px LR in (64 px + a halo of 4) -> 128 px HR out, 8 frames each
val by content: photo 784, video 1,585, screen 1,154


## 2. The published graphs

The models are the graphs notebooks 4, 10 and 12 published, run the way they ship. A graph's input
is a client tile (a 320x180 step and its halo), so the patches are laid side by side in it, 8 to
a tile, and each one's output is cut back out. A patch's core sees nothing of its neighbours,
since no model reaches past the halo of 4 - and every graph has to score here what its notebook
measured in torch, or this notebook stops.

In [3]:
SESSIONS = {name: webexport.session(webexport.MODELS_DIR / file) for name, (file, _) in GRAPHS.items()}


def run_graph(name, lr):
    """A published graph on LR patches (N, 3, 72, 72), as it ships. Returns the HR cores (N, 3, 128, 128)."""
    runner = SESSIONS[name]
    height, width = runner.get_inputs()[0].shape[2:]
    rows, cols = height // PATCH, width // PATCH
    lr = lr.cpu().numpy()
    out = np.zeros((len(lr), 3, PATCH * SCALE, PATCH * SCALE), np.float32)
    for first in range(0, len(lr), rows * cols):
        tile = np.zeros((1, 3, height, width), np.float32)
        group = lr[first:first + rows * cols]
        for k, patch in enumerate(group):
            r, c = divmod(k, cols)
            tile[0, :, r * PATCH:(r + 1) * PATCH, c * PATCH:(c + 1) * PATCH] = patch
        result = webexport.run(runner, tile)
        for k in range(len(group)):
            r, c = divmod(k, cols)
            out[first + k] = result[0, :, r * PATCH * SCALE:(r + 1) * PATCH * SCALE, c * PATCH * SCALE:(c + 1) * PATCH * SCALE]
    return trim(torch.from_numpy(out).to(DEVICE))


@torch.no_grad()
def scores_of(make, batch=256):
    """Luma SSIM and PSNR of every validation patch, as (N,) tensors on the CPU, for the
    pictures `make(lr, index)` makes from a batch of LR patches and their indices."""
    parts = []
    for first in range(0, len(val_lr), batch):
        index = np.arange(first, min(first + batch, len(val_lr)))
        parts.append(metrics.per_image(make(to_tensor(val_lr[index]), index), to_tensor(val_hr[index])))
    return {key: torch.cat([part[key] for part in parts]).cpu() for key in ("ssim", "y")}


def summary(scores):
    """Mean luma SSIM and PSNR over every patch, and SSIM by content."""
    row = {"ssim": scores["ssim"].mean().item(), "y": scores["y"].mean().item()}
    return row | {kind: scores["ssim"][index].mean().item() for kind, index in by_content.items()}


def show(rows, extra=None):
    """A table of {name: per-patch scores}; `extra(name)` may add a last column."""
    width = max(len(name) for name in rows) + 2
    print(f"{'':<{width}}{'ssim-y':>8}{'psnr-y':>8}" + "".join(f"{kind:>9}" for kind in by_content))
    for name, scores in rows.items():
        s = summary(scores)
        print(f"{name:<{width}}{s['ssim']:8.4f}{s['y']:8.2f}" + "".join(f"{s[kind]:9.4f}" for kind in by_content)
              + (extra(name) if extra else ""))


per = {"bicubic": scores_of(lambda lr, index: trim(models.bicubic(lr, SCALE)))}
outputs = {}   # every graph's HR cores, kept on the CPU: the post-filters' input
for name in GRAPHS:
    outputs[name] = torch.cat([run_graph(name, to_tensor(val_lr[first:first + 256])).cpu()
                               for first in range(0, len(val_lr), 256)])
    per[name] = scores_of(lambda lr, index, name=name: outputs[name][index].to(DEVICE))
for name, (_, recorded) in GRAPHS.items():
    measured = per[name]["ssim"].mean().item()
    assert abs(measured - recorded) < 5e-4, f"{name}: {measured:.4f} as published, {recorded:.4f} in its notebook"
print("every graph scores what its notebook measured; ssim-y overall and by content:")
show(per)

every graph scores what its notebook measured; ssim-y overall and by content:
                ssim-y  psnr-y    photo    video   screen
bicubic         0.9049   33.18   0.8359   0.9535   0.8851
notebook 4      0.9292   34.45   0.8576   0.9657   0.9277
SDY x2 table    0.9343   34.63   0.8526   0.9637   0.9495
DirNet D        0.9303   34.22   0.8520   0.9636   0.9378


## 3. Routing

**A classifier per tile.** A few numbers say a lot about whether a tile is a drawn screen: how
often a pixel is exactly its neighbour (flat fills), how many edges are sharp, how many steps are
gentle (gradients), how many colours it has. A logistic regression on them, fitted on every 2nd
training patch, is the classifier - on the HR tile, where the host sees exact flat fills and
crisp edges, and on the LR tile, which the client has for free but through the codec.

**The routes.** A tile labelled a screen goes to a screen model (notebook 10's table, or notebook
12's D), any other to notebook 4's model; the label is the true one, the host's or the client's.
And the ceiling of routing: the host runs all three on every tile and sends which one came
closest to its HR frame. A label costs 1 bit a tile and the best of three log2(3) - about 127
and 201 bits a frame, under a thousandth of it.

In [4]:
def features(rgb):
    """What tells a screen from a photo or a video, per tile, from 8-bit RGB values (N, 3, h, w):
    how often a pixel is exactly its right and its lower neighbour, the share of sharp edges, the
    mean step, the share of gentle steps, and how many colours there are."""
    y = luma(rgb)[:, 0]
    same_across = (rgb[..., 1:] == rgb[..., :-1]).all(dim=1).float().mean(dim=(1, 2))
    same_down = (rgb[..., 1:, :] == rgb[..., :-1, :]).all(dim=1).float().mean(dim=(1, 2))
    across, down = (y[..., 1:] - y[..., :-1]).abs(), (y[..., 1:, :] - y[..., :-1, :]).abs()
    sharp = ((across > 48).float().mean(dim=(1, 2)) + (down > 48).float().mean(dim=(1, 2))) / 2
    step = (across.mean(dim=(1, 2)) + down.mean(dim=(1, 2))) / 64
    gentle = ((across > 0) & (across < 4)).float().mean(dim=(1, 2))
    packed = (rgb[:, 0].long() << 16) | (rgb[:, 1].long() << 8) | rgb[:, 2].long()
    colours = torch.tensor([len(torch.unique(tile)) for tile in packed.flatten(1)], device=rgb.device) / packed[0].numel()
    return torch.stack([same_across, same_down, sharp, step, gentle, colours], dim=1)


def fit_classifier(x, screen):
    """A logistic regression on standardised features: above 0 means a screen."""
    mean, std = x.mean(dim=0), x.std(dim=0) + 1e-6
    x = (x - mean) / std
    weights = torch.zeros(x.shape[1] + 1, device=DEVICE, requires_grad=True)
    optimiser = torch.optim.LBFGS([weights], max_iter=500)

    def loss():
        optimiser.zero_grad()
        value = (F.binary_cross_entropy_with_logits(x @ weights[:-1] + weights[-1], screen.float())
                 + 1e-3 * weights[:-1].pow(2).sum())
        value.backward()
        return value

    optimiser.step(loss)
    weights = weights.detach()
    return lambda z: ((z - mean) / std) @ weights[:-1] + weights[-1] > 0


train_set = PatchPairs(DATA_DIR, "train", manifest, augment=False)
train_index = np.arange(0, len(train_set), TRAIN_STEP)
_, train_screen = per_patch("train")
train_screen = torch.from_numpy(train_screen[train_index]).to(DEVICE)
host_train, client_train = [], []
with torch.no_grad():
    for lr, hr in DataLoader(Subset(train_set, train_index), batch_size=256):
        host_train.append(features((hr.to(DEVICE) * 255).round()))
        client_train.append(features((patches.trim(lr.to(DEVICE), HALO) * 255).round()))
    host_val = torch.cat([features((to_tensor(val_hr[i:i + 256]) * 255).round()) for i in range(0, len(val_hr), 256)])
    client_val = torch.cat([features((to_tensor(val_lr[i:i + 256, HALO:-HALO, HALO:-HALO]) * 255).round())
                            for i in range(0, len(val_lr), 256)])

truth = torch.from_numpy(val_screen).to(DEVICE)
labels = {"true": truth,
          "host's": fit_classifier(torch.cat(host_train), train_screen)(host_val),
          "client's": fit_classifier(torch.cat(client_train), train_screen)(client_val)}
for name in ("host's", "client's"):
    guess = labels[name]
    print(f"{name:<9} classifier: {(guess == truth).float().mean():.1%} right; finds {(guess & truth).sum() / truth.sum():.1%} "
          f"of the screens; {(guess & truth).sum() / guess.sum():.1%} of what it calls a screen is one")

host's    classifier: 90.3% right; finds 94.3% of the screens; 79.8% of what it calls a screen is one
client's  classifier: 87.3% right; finds 89.9% of the screens; 75.7% of what it calls a screen is one


In [5]:
def routed(label, screen_model):
    """Per-patch scores when a tile labelled a screen goes to `screen_model` and any other to BEST."""
    label = label.cpu()
    return {key: torch.where(label, per[screen_model][key], per[BEST][key]) for key in ("ssim", "y")}


routes, route_bits = {name: per[name] for name in GRAPHS}, {}
for screen_model in ("SDY x2 table", "DirNet D"):
    for name, label in labels.items():
        route = f"screens to {screen_model}, {name} label"
        routes[route] = routed(label, screen_model)
        route_bits[route] = TILES_PER_FRAME if name != "client's" else 0
# the host runs all three and sends which came closest to its HR frame
choice = torch.stack([per[name]["ssim"] for name in GRAPHS]).argmax(dim=0)
routes["the best of the three per tile, host's choice"] = {
    key: torch.stack([per[name][key] for name in GRAPHS]).gather(0, choice[None])[0] for key in ("ssim", "y")}
route_bits["the best of the three per tile, host's choice"] = TILES_PER_FRAME * math.log2(len(GRAPHS))
show(routes, extra=lambda name: f"{route_bits[name]:8.0f} bits a frame" if name in route_bits else "")

                                                 ssim-y  psnr-y    photo    video   screen
notebook 4                                       0.9292   34.45   0.8576   0.9657   0.9277
SDY x2 table                                     0.9343   34.63   0.8526   0.9637   0.9495
DirNet D                                         0.9303   34.22   0.8520   0.9636   0.9378
screens to SDY x2 table, true label              0.9363   34.97   0.8576   0.9657   0.9495     127 bits a frame
screens to SDY x2 table, host's label            0.9364   34.91   0.8575   0.9657   0.9498     127 bits a frame
screens to SDY x2 table, client's label          0.9361   34.89   0.8576   0.9656   0.9489       0 bits a frame
screens to DirNet D, true label                  0.9325   34.57   0.8576   0.9657   0.9378     127 bits a frame
screens to DirNet D, host's label                0.9326   34.56   0.8575   0.9658   0.9382     127 bits a frame
screens to DirNet D, client's label              0.9324   34.53   0.8575   0

## 4. Filters fitted by the host

`helpers/postfilter.py` holds the machinery; notebook 14 uses it on whole frames.

**Classes**, from what the client has. At every LR pixel, how much luma bends across, down and
along both diagonals over its 3x3 neighbourhood (1-D Laplacians, as VVC's loop filter measures
them). A direction is taken if it bends at least twice as much as the one at right angles to it
(ALF's rule), else none; activity is the bend across and down together, in 5 levels at the
quantiles of the training set. 5 x 5 = 25 classes.

**Filters.** On bicubic, a filter makes one of the four sub-pixels' luma from the 5x5 LR luma
around its pixel, plus an offset: 26 coefficients, four filters per class. On a model, a
post-filter makes a pixel's luma from the 5x5 of the model's output around it: 26 coefficients,
one filter per class, the class of its LR pixel. Either way the change in luma is added to R, G
and B, as every luma model here does, so colour stays as it was.

**Fits.** Least squares against the HR luma, all of it closed-form:

- **zero bits** - fitted once on every 2nd training patch, and built into the client;
- **per source, from the HR** - the host's fit, on the other 7 frames of the source's patches,
  pulled toward the zero-bit filter with the weight of one pixel. A class the source shows on
  fewer than 2,000 pixels keeps the zero-bit filter, and a flag per class says which.

**Bits.** A filter sent is its difference from the zero-bit filter, in 10-bit steps of its own
size, plus the step (16 bits) - and it is scored after that rounding, as the client would get it.
Each kind comes as **one filter** for everything or **25 classes**. On every published model:
whether a post-filter adds anything depends on what the model already gets right.

In [6]:
BASES = {"bicubic": SCALE * SCALE, **{name: 1 for name in GRAPHS}}   # what a filter goes on -> its outputs per pixel


def no_group(n):
    """Group 0 for all `n` patches: the filters fitted on everything."""
    return torch.zeros(n, dtype=torch.long, device=DEVICE)


def core_classes(lr, levels):
    """The class of every LR core pixel: (N, 64, 64)."""
    return postfilter.classes(luma(lr), levels)[:, HALO:HALO + CORE, HALO:HALO + CORE]


def lr_windows(lr):
    """The 5x5 LR luma around every core pixel, and a 1 for the offset: (N, 26, 64 * 64)."""
    return postfilter.windows(luma(lr)[..., HALO - 2:HALO + CORE + 2, HALO - 2:HALO + CORE + 2], TAPS, pad=False)


def sub_pixels(picture):
    """The luma of an HR core sub-pixel by sub-pixel, in the PixelShuffle's order: (N, 4, 64 * 64)."""
    return F.pixel_unshuffle(luma(picture), SCALE).flatten(2)


def add(fit, base, lr, hr, picture, group):
    """Add a batch to a fit: on bicubic, LR windows against the HR sub-pixels; on a model, windows
    of its output `picture` against the HR luma."""
    pixel_class = core_classes(lr, levels)
    if base == "bicubic":
        fit.add(lr_windows(lr), sub_pixels(hr), pixel_class.flatten(1), group)
    else:
        fit.add(postfilter.windows(luma(picture), TAPS), luma(hr).flatten(2),
                postfilter.upsampled(pixel_class, SCALE).flatten(1), group)

### The fits

The zero-bit filters from every 2nd training patch, then the host's from every validation source's
other 7 frames. Every model runs on every one of those frames, as published - the host would run
the client's model on its own decoded frames to fit a post-filter on it.

In [7]:
with torch.no_grad():
    sample = torch.cat([luma(lr.to(DEVICE))[..., HALO:HALO + CORE, HALO:HALO + CORE]
                        for lr, _ in DataLoader(Subset(train_set, train_index[::8]), batch_size=256)])
    levels = postfilter.activity_levels(sample, LEVELS)

    # zero bits: on the training patches
    general = {base: postfilter.Fit(1, CLASSES, out, TAPS, DEVICE) for base, out in BASES.items()}
    for lr, hr in DataLoader(Subset(train_set, train_index), batch_size=128):
        lr, hr = lr.to(DEVICE), hr.to(DEVICE)
        for base, fit in general.items():
            add(fit, base, lr, hr, None if base == "bicubic" else run_graph(base, lr), no_group(len(lr)))

    # per source: on the other 7 frames of each validation patch
    host = {base: postfilter.Fit(len(sources), CLASSES, out, TAPS, DEVICE) for base, out in BASES.items()}
    all_lr = np.load(DATA_DIR / manifest["splits"]["val"]["lr"], mmap_mode="r")   # (N, frames, 72, 72, 3)
    all_hr = np.load(DATA_DIR / manifest["splits"]["val"]["hr"], mmap_mode="r")
    frames = all_lr.shape[1]
    scored = np.arange(len(all_lr)) % frames   # the frame every notebook scores
    for first in range(0, len(all_lr), 64):
        index = np.arange(first, min(first + 64, len(all_lr)))
        chunk_lr, chunk_hr = all_lr[index], all_hr[index]
        for frame in range(frames):
            other = scored[index] != frame
            lr, hr = to_tensor(chunk_lr[other, frame]), to_tensor(chunk_hr[other, frame])
            group = torch.from_numpy(val_source[index[other]]).to(DEVICE)
            for base, fit in host.items():
                add(fit, base, lr, hr, None if base == "bicubic" else run_graph(base, lr), group)

pixels = host["bicubic"].pixels()
print(f"activity levels {[round(v, 4) for v in levels.tolist()]}; training pixels per class "
      f"{int(general['bicubic'].pixels().min()):,} to {int(general['bicubic'].pixels().max()):,}; per source, "
      f"{(pixels >= MIN_PIXELS).sum(dim=1).float().mean():.1f} of {CLASSES} classes seen often enough to send")

activity levels [0.0006, 0.0111, 0.04, 0.1266]; training pixels per class 63,699 to 6,313,003; per source, 17.4 of 25 classes seen often enough to send


In [8]:
filters, kbit = {}, {}
for base, out in BASES.items():
    for one_class, size in ((True, "one filter"), (False, "25 classes")):
        zero = general[base].solve(one_class)                                   # (1, classes, 26, outputs)
        prior = zero.expand(len(sources), -1, -1, -1)
        fitted = host[base].solve(one_class, prior, RIDGE)
        if one_class:   # one filter, always sent, and no flags
            sent = torch.zeros(len(sources), CLASSES, dtype=torch.bool, device=DEVICE)
            sent[:, 0] = True
            bits = postfilter.update_bits(sent, out, TAPS, COEFFICIENT_BITS, STEP_BITS) - CLASSES
        else:
            sent = host[base].pixels() >= MIN_PIXELS
            bits = postfilter.update_bits(sent, out, TAPS, COEFFICIENT_BITS, STEP_BITS)
        kbit[base, size] = bits.double().mean().item() / 1e3
        received = torch.where(sent[..., None, None] | one_class, postfilter.quantise(fitted, prior, COEFFICIENT_BITS), prior)
        filters[base, size, "zero bits"] = zero.float()
        filters[base, size, "per source"] = received.float()


def filtered(base, size, fit):
    """A `make` for `scores_of`: the base picture with its luma changed by these filters."""
    chosen, one_class = filters[base, size, fit], size == "one filter"

    def make(lr, index):
        group = torch.from_numpy(val_source[index]).to(DEVICE) if fit == "per source" else no_group(len(lr))
        pixel_class = core_classes(lr, levels)
        if one_class:
            pixel_class = torch.zeros_like(pixel_class)
        if base == "bicubic":
            picture = trim(models.bicubic(lr, SCALE))
            change = postfilter.through(chosen, lr_windows(lr), pixel_class.flatten(1), group) - sub_pixels(picture)
            return picture + F.pixel_shuffle(change.view(len(lr), SCALE * SCALE, CORE, CORE), SCALE)
        picture = outputs[base][index].to(DEVICE)
        return postfilter.post_filter(picture, chosen, postfilter.upsampled(pixel_class, SCALE), group, TAPS)

    return make


results, result_kbit = {"bicubic": per["bicubic"]}, {}
for base in BASES:
    if base != "bicubic":
        results[base] = per[base]
    for size in ("one filter", "25 classes"):
        for fit in ("zero bits", "per source"):
            name = f"{base} + {size}, {fit}"
            results[name] = scores_of(filtered(base, size, fit))
            result_kbit[name] = kbit[base, size] if fit == "per source" else 0.0
show(results, extra=lambda name: f"{result_kbit[name]:8.1f} kbit an update" if name in result_kbit else "")

                                         ssim-y  psnr-y    photo    video   screen
bicubic                                  0.9049   33.18   0.8359   0.9535   0.8851
bicubic + one filter, zero bits          0.9091   33.47   0.8432   0.9517   0.8953     0.0 kbit an update
bicubic + one filter, per source         0.9142   33.94   0.8518   0.9567   0.8981     1.1 kbit an update
bicubic + 25 classes, zero bits          0.9144   33.69   0.8462   0.9585   0.9001     0.0 kbit an update
bicubic + 25 classes, per source         0.9239   34.60   0.8580   0.9642   0.9133    19.2 kbit an update
notebook 4                               0.9292   34.45   0.8576   0.9657   0.9277
notebook 4 + one filter, zero bits       0.9290   34.50   0.8564   0.9656   0.9280     0.0 kbit an update
notebook 4 + one filter, per source      0.9332   35.02   0.8614   0.9685   0.9333     0.3 kbit an update
notebook 4 + 25 classes, zero bits       0.9293   34.43   0.8559   0.9660   0.9287     0.0 kbit an update
notebook 

## 5. What the bits are worth

**The same bits in H.264.** Each source's bitrate was drawn at random (log-uniform, 0.2-1.0 bpp),
so how a model's score moves with it across sources - what a source shows held apart - measures
what more bits in the video buy. Coded sources only; the error is the regression's standard
error, and it is not small.

**Every hint side by side**, against what the client gets without it: routing by the host's
label against routing by its own, the host's choice of model against the client's routing, and
each host-fitted filter against the same filter fitted on training data. A hint's bits are shown
per update and as a share of one frame of the default stream - its cost if it were sent again
with every frame, which on a still desktop it is not - and next to that, what the same share of
bits would buy in H.264.

In [9]:
def bitrate_slope(scores):
    """What doubling the bitrate buys `scores` (per patch), and its standard error: a regression of
    each coded source's mean on log2 of its bitrate, with screens and video apart from photos."""
    rows = []
    for s in sources:
        if s["degradation"]["bpp"] is not None:
            first, end = s["patches"]
            rows.append((1.0, math.log2(s["degradation"]["bpp"]), content(s) == "screen", content(s) == "video",
                         scores[first:end].mean().item()))
    data = np.array(rows, float)
    x, y = data[:, :4], data[:, 4]
    beta = np.linalg.lstsq(x, y, rcond=None)[0]
    noise = ((y - x @ beta) ** 2).sum() / (len(y) - x.shape[1])
    return beta[1], math.sqrt(noise * np.linalg.inv(x.T @ x)[1, 1])


slope, slope_error = bitrate_slope(per[BEST]["ssim"])
db_slope, db_error = bitrate_slope(per[BEST]["y"])
print(f"doubling the bitrate buys notebook 4's model {slope:+.4f} ± {slope_error:.4f} ssim-y and "
      f"{db_slope:+.2f} ± {db_error:.2f} dB luma ({sum(s['degradation']['bpp'] is not None for s in sources)} coded sources)\n")

HINTS = {   # hint -> (what the client has without it, bits an update)
    "screens to SDY x2 table, host's label": ("screens to SDY x2 table, client's label", TILES_PER_FRAME),
    "screens to DirNet D, host's label": ("screens to DirNet D, client's label", TILES_PER_FRAME),
    "the best of the three per tile, host's choice": ("screens to SDY x2 table, client's label",
                                                      TILES_PER_FRAME * math.log2(len(GRAPHS))),
    **{f"{base} + {size}, per source": (f"{base} + {size}, zero bits", kbit[base, size] * 1e3)
       for base in ("bicubic", *GRAPHS) for size in ("one filter", "25 classes")},
}
everything = routes | results
width = max(len(name) for name in HINTS) + 2
print(f"{'':<{width}}{'ssim-y':>8}{'gain':>9}{'psnr-y':>8}{'gain':>7}{'kbit':>8}{'of a frame':>12}{'in H.264':>10}")
for name, (without, bits) in HINTS.items():
    s, w = summary(everything[name]), summary(everything[without])
    share = bits / FRAME_BITS
    print(f"{name:<{width}}{s['ssim']:8.4f}{s['ssim'] - w['ssim']:+9.4f}{s['y']:8.2f}{s['y'] - w['y']:+7.2f}"
          f"{bits / 1e3:8.1f}{share:12.2%}{slope * math.log2(1 + share):+10.4f}")
print("\ngain: over the same client without the hint; in H.264: what the same bits buy, if sent with every frame")

doubling the bitrate buys notebook 4's model +0.0143 ± 0.0062 ssim-y and +1.39 ± 0.59 dB luma (204 coded sources)

                                                 ssim-y     gain  psnr-y   gain    kbit  of a frame  in H.264
screens to SDY x2 table, host's label            0.9364  +0.0004   34.91  +0.02     0.1       0.05%   +0.0000
screens to DirNet D, host's label                0.9326  +0.0002   34.56  +0.02     0.1       0.05%   +0.0000
the best of the three per tile, host's choice    0.9370  +0.0010   35.04  +0.15     0.2       0.08%   +0.0000
bicubic + one filter, per source                 0.9142  +0.0051   33.94  +0.47     1.1       0.46%   +0.0001
bicubic + 25 classes, per source                 0.9239  +0.0095   34.60  +0.91    19.2       8.05%   +0.0016
notebook 4 + one filter, per source              0.9332  +0.0042   35.02  +0.52     0.3       0.12%   +0.0000
notebook 4 + 25 classes, per source              0.9358  +0.0065   35.30  +0.87     6.1       2.58%   +0.0005
SDY x